# Alpha-u abundant-data broad benchmark

Clean driver for the alpha-parameterization simplification experiment. It executes the committed abundant-data benchmark and changes only the nonlinear identifier to `identify_nonlinear_unbounded`, fingerprints that source in the scientific config, and uses the short internal namespace `alpha_u_ab`.


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

# Thin ablation driver: the committed abundant-data notebook is the source of
# truth.  Only the identifier and alpha-ablation filesystem/config identity are
# changed.
HERE = Path.cwd()
BASE_NOTEBOOK = (
    HERE
    / "experiment_2026_08_24_abundant_all_dynamics_multitopology_multiseed.ipynb"
)
if not BASE_NOTEBOOK.exists():
    raise FileNotFoundError(f"Missing abundant-data base notebook: {BASE_NOTEBOOK}")

print("Alpha-u abundant base:", BASE_NOTEBOOK.name)

base_nb = json.loads(BASE_NOTEBOOK.read_text(encoding="utf-8"))

patched_identifier = False
patched_study = False
patched_results = False
patched_identifier_hash = False
patched_scientific_config = False

patched_cells: list[tuple[int, str]] = []

for cell_index, cell in enumerate(base_nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))

    # 1) Direct abundant-data identifier import -> unbounded-alpha identifier.
    if "from opinion_dynamics.identify_nonlinear import (" in src:
        src = src.replace(
            "from opinion_dynamics.identify_nonlinear import (",
            "from opinion_dynamics.identify_nonlinear_unbounded import (",
            1,
        )
        patched_identifier = True

    # Add an exact source fingerprint once GraphIdentifierEnv is available.
    online_anchor = (
        "import opinion_dynamics.experiments.online_single_shot as online_single_shot\n"
    )
    if (
        online_anchor in src
        and "IDENTIFIER_SOURCE_SHA256" not in src
    ):
        src = src.replace(
            online_anchor,
            online_anchor
            + "IDENTIFIER_SOURCE_PATH = Path(inspect.getsourcefile(GraphIdentifierEnv))\n"
            + "IDENTIFIER_SOURCE_SHA256 = hashlib.sha256("
              "IDENTIFIER_SOURCE_PATH.read_bytes()).hexdigest()\n",
            1,
        )
        patched_identifier_hash = True

    # 2) Short internal cache/results namespace.
    new_src, n = re.subn(
        r'STUDY_NAME\s*=\s*"[^"]+"',
        'STUDY_NAME = "alpha_u_ab"',
        src,
        count=1,
    )
    if n:
        src = new_src
        patched_study = True

    new_src, _ = re.subn(
        r'PIPELINE_VERSION\s*=\s*"[^"]+"',
        'PIPELINE_VERSION = "2026-09-06-alpha-u-ab-v1"',
        src,
        count=1,
    )
    src = new_src

    old_results = "experiment_2026_08_24_abundant_all_dynamics_multitopology_multiseed"
    if old_results in src:
        src = src.replace(old_results, "alpha_u_ab")
        patched_results = True

    # 3) Make both the parameterization and exact identifier source part of
    # the abundant-data config hash.
    if "SCIENTIFIC_CONFIG = {" in src:
        additions = ""
        if '"alpha_parameterization"' not in src:
            additions += (
                '    "alpha_parameterization": '
                '"softplus_over_log2_positive_unbounded",\n'
            )
            patched_scientific_config = True
        if '"identifier_source_sha256"' not in src:
            additions += (
                '    "identifier_source_sha256": IDENTIFIER_SOURCE_SHA256,\n'
            )
        if additions:
            src = src.replace(
                "SCIENTIFIC_CONFIG = {\n",
                "SCIENTIFIC_CONFIG = {\n" + additions,
                1,
            )

    patched_cells.append((cell_index, src))

required = {
    "unbounded identifier import": patched_identifier,
    "short STUDY_NAME": patched_study,
    "short RESULTS_DIR": patched_results,
    "identifier source fingerprint": patched_identifier_hash,
    "alpha scientific-config marker": patched_scientific_config,
}
missing = [name for name, ok in required.items() if not ok]
if missing:
    raise RuntimeError(
        "Refusing to execute: the committed abundant notebook did not match the "
        "expected structure. Missing patches: " + ", ".join(missing)
    )

print("Alpha-u overrides validated:")
for name in required:
    print("  OK:", name)

g = globals()
for cell_index, src in patched_cells:
    print(f"[base code cell {cell_index}]")
    exec(
        compile(src, f"{BASE_NOTEBOOK.name}:cell_{cell_index}", "exec"),
        g,
        g,
    )
